In [1]:
!pip install xgboost imbalanced-learn scikit-optimize joblib

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import f1_score, accuracy_score, classification_report

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from skopt import BayesSearchCV
from skopt.space import Integer, Real

import joblib

print("🚧 Pipeline v3 ready (imports complete)")

🚧 Pipeline v3 ready (imports complete)


In [2]:
df = pd.read_csv("../../research/datasets/delay-cases/delay_data.csv")

feature_cols = [
    "phase_group", "sub_phase", "district", "province",
    "floors", "delay_category", "labour_availability",
    "material_supply", "weather_severity", "cumulative_delay"
]

target = "delay_risk"

df = df.dropna(subset=feature_cols + [target])

X = df[feature_cols]
y = df[target]

print(f"✅ Data loaded | Shape: {df.shape}")
print(df[target].value_counts())

✅ Data loaded | Shape: (192, 15)
delay_risk
MEDIUM    103
HIGH       60
LOW        29
Name: count, dtype: int64


In [3]:
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

print("✅ Target encoded:")
print(list(target_encoder.classes_))

✅ Target encoded:
['HIGH', 'LOW', 'MEDIUM']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("✅ Split done")
print("Train:", X_train.shape)
print("Test:", X_test.shape)

✅ Split done
Train: (153, 10)
Test: (39, 10)


In [5]:
cat_features = [
    "phase_group", "sub_phase",
    "district", "province", "delay_category"
]

num_features = [
    "floors", "labour_availability",
    "material_supply", "weather_severity",
    "cumulative_delay"
]

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, cat_features),
        ("num", numeric_transformer, num_features)
    ]
)

print("✅ Preprocessing ready")

✅ Preprocessing ready


In [6]:
model = XGBClassifier(
    eval_metric="mlogloss",
    random_state=42
)

pipeline = ImbPipeline(steps=[
    ("preprocess", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", model)
])

print("🚀 Pipeline built (SMOTE inside CV safe)")

🚀 Pipeline built (SMOTE inside CV safe)


In [7]:
search_space = {
    "model__n_estimators": Integer(100, 400),
    "model__max_depth": Integer(3, 8),
    "model__learning_rate": Real(0.01, 0.2),
    "model__subsample": Real(0.6, 1.0),
    "model__colsample_bytree": Real(0.6, 1.0),
    "model__min_child_weight": Integer(1, 10)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("🚀 Bayesian Optimization started...")

bayes_search = BayesSearchCV(
    estimator=pipeline,
    search_spaces=search_space,
    n_iter=8,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1,
    random_state=42
)

bayes_search.fit(X_train, y_train)

print("✅ Optimization complete")
print("Best CV F1:", round(bayes_search.best_score_, 4))
print("Best Params:", bayes_search.best_params_)

🚀 Bayesian Optimization started...
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
✅ Optimization complete
Best CV F1: 0.8815
Best Params: OrderedDict({'model__colsample_bytree': 0.817361227076125, 'model__learning_rate': 0.18480175302309013, 'model__max_depth': 5, 'model__min_child_weight': 9, 'model__n_estimators': 197, 'model__subsample': 0.6204538214708207})


In [8]:
best_model = bayes_search.best_estimator_
print("✅ Best model extracted")

✅ Best model extracted


In [9]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report
)

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")

print("📊 FINAL MODEL PERFORMANCE")
print("━━━━━━━━━━━━━━━━━━━━━━")
print(f"Accuracy : {accuracy:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print("━━━━━━━━━━━━━━━━━━━━━━")

print("\n📄 Classification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=target_encoder.classes_
))

📊 FINAL MODEL PERFORMANCE
━━━━━━━━━━━━━━━━━━━━━━
Accuracy : 0.8718
F1 Score : 0.8716
Precision: 0.8730
Recall   : 0.8718
━━━━━━━━━━━━━━━━━━━━━━

📄 Classification Report:

              precision    recall  f1-score   support

        HIGH       0.91      0.83      0.87        12
         LOW       0.83      0.83      0.83         6
      MEDIUM       0.86      0.90      0.88        21

    accuracy                           0.87        39
   macro avg       0.87      0.86      0.86        39
weighted avg       0.87      0.87      0.87        39



In [10]:
import joblib
import os

os.makedirs("../../performance/models", exist_ok=True)

joblib.dump({
    "model": best_model,
    "target_encoder": target_encoder
}, "../../performance/models/delay_risk_pipeline_v2.pkl")

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(" Production Pipeline Saved")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("Path:")
print("../../performance/models/delay_risk_pipeline_v2.pkl")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Production Pipeline Saved
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Path:
../../performance/models/delay_risk_pipeline_v2.pkl
━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [11]:
import joblib
import pandas as pd

loaded = joblib.load(
    "../../performance/models/delay_risk_pipeline_v2.pkl"
)

model = loaded["model"]
encoder = loaded["target_encoder"]

sample = pd.DataFrame([{
    "phase_group": "Foundations",
    "sub_phase": "Foundation work",
    "district": "Jaffna",
    "province": "Northern Province",
    "floors": 2,
    "delay_category": "Labour",
    "labour_availability": 0.30,
    "material_supply": 1,
    "weather_severity": 0.75,
    "cumulative_delay": 20
}])

pred = model.predict(sample)
proba = model.predict_proba(sample)

risk_label = encoder.inverse_transform(pred)[0]
confidence = round(float(max(proba[0])) * 100, 2)

print("━━━━━━━━━━━━━━━━━━━━━━━")
print("  Production Test Result")
print("━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Risk:       {risk_label}")
print(f"  Confidence: {confidence}%")
print("━━━━━━━━━━━━━━━━━━━━━━━")
print(" Pipeline working correctly")

━━━━━━━━━━━━━━━━━━━━━━━
  Production Test Result
━━━━━━━━━━━━━━━━━━━━━━━
  Risk:       MEDIUM
  Confidence: 73.72%
━━━━━━━━━━━━━━━━━━━━━━━
 Pipeline working correctly
